# Phân tích bán hàng với PySpark

## Sales Analytics with PySpark – UCI Online Retail II

**Mục tiêu:** xử lý hơn 1 triệu giao dịch và chuyển dữ liệu thành KPI, insight và đề xuất kinh doanh.

## Dataset
- **UCI Online Retail II**
- 1,067,371 giao dịch trong hai năm
- Nguồn: https://archive.ics.uci.edu/dataset/502/online+retail+ii
- DOI: 10.24432/C5CG6D
- License: CC BY 4.0

> Dataset gốc là Excel gồm hai sheet. Phần setup dưới đây tải và chuyển từng sheet sang CSV trước khi Spark đọc.

## Cài đặt và khởi tạo Spark

In [ ]:
!pip -q install pyspark pandas openpyxl

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *

spark = (SparkSession.builder
         .appName('SalesAnalyticsOnlineRetailII')
         .getOrCreate())

print('Spark version:', spark.version)

## Tải dữ liệu từ UCI và chuyển Excel sang CSV
Chạy cell này trên Colab/Jupyter có Internet. Nếu đã có `online_retail_II.csv`, có thể bỏ qua.

In [ ]:
import os, zipfile, urllib.request, pandas as pd

URL = 'https://archive.ics.uci.edu/static/public/502/online%2Bretail%2Bii.zip'
ZIP_PATH = 'online_retail_II.zip'
EXTRACT_DIR = 'online_retail_II_raw'
CSV_PATH = 'online_retail_II.csv'

if not os.path.exists(CSV_PATH):
    if not os.path.exists(ZIP_PATH):
        urllib.request.urlretrieve(URL, ZIP_PATH)
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)

    xlsx_files = [os.path.join(EXTRACT_DIR, f) for f in os.listdir(EXTRACT_DIR) if f.lower().endswith('.xlsx')]
    assert xlsx_files, 'Không tìm thấy file .xlsx'
    xlsx_path = xlsx_files[0]
    book = pd.ExcelFile(xlsx_path)
    first = True
    for sheet in book.sheet_names:
        print('Converting:', sheet)
        pdf = pd.read_excel(xlsx_path, sheet_name=sheet)
        pdf.to_csv(CSV_PATH, mode='w' if first else 'a', index=False, header=first)
        first = False
    print('Created:', CSV_PATH)
else:
    print('Found:', CSV_PATH)

## 1. Data Understanding

In [ ]:
sales_raw = spark.read.option('header', True).option('inferSchema', True).csv('online_retail_II.csv')
sales_raw.show(10, truncate=False)
sales_raw.printSchema()

In [ ]:
print('Rows:', sales_raw.count())
print('Columns:', len(sales_raw.columns))
print(sales_raw.columns)

### Bài tập 1. Khám phá dữ liệu
1. Đếm số hóa đơn, sản phẩm, khách hàng và quốc gia phân biệt.
2. Xác định min/max InvoiceDate.
3. Giải thích vì sao số dòng lớn hơn số hóa đơn.

In [ ]:
sales_raw.select(
    F.countDistinct('Invoice').alias('Invoices') if 'Invoice' in sales_raw.columns else F.countDistinct('InvoiceNo').alias('Invoices')
).show()

> **Lưu ý:** UCI Online Retail II có thể dùng tên cột `Invoice`, `Price`, `Customer ID` thay vì `InvoiceNo`, `UnitPrice`, `CustomerID` tùy phiên bản file. Cell sau chuẩn hóa tên cột để phần còn lại của notebook thống nhất.

In [ ]:
rename_map = {
    'Invoice':'InvoiceNo',
    'Price':'UnitPrice',
    'Customer ID':'CustomerID'
}
df = sales_raw
for old, new in rename_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)
print(df.columns)

## 2. Kiểm tra missing values và dữ liệu bất thường

In [ ]:
missing = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns])
missing.show(truncate=False)

In [ ]:
df.select(
    F.sum((F.col('Quantity') <= 0).cast('int')).alias('Quantity_le_0'),
    F.sum((F.col('UnitPrice') <= 0).cast('int')).alias('UnitPrice_le_0'),
    F.sum(F.col('InvoiceNo').cast('string').startswith('C').cast('int')).alias('CancelledRows')
).show()

### Bài tập 2. Data Cleaning
Tạo `sales_clean` với các giao dịch bán hợp lệ và `sales_customer` chỉ gồm các dòng có CustomerID.

In [ ]:
df = df.withColumn('InvoiceDate', F.to_timestamp('InvoiceDate'))

df = df.withColumn('IsCancelled', F.col('InvoiceNo').cast('string').startswith('C'))

sales_clean = df.filter(
    (~F.col('IsCancelled')) &
    (F.col('Quantity') > 0) &
    (F.col('UnitPrice') > 0)
)

sales_customer = sales_clean.filter(F.col('CustomerID').isNotNull())

print('Raw rows:', df.count())
print('Clean rows:', sales_clean.count())
print('Customer rows:', sales_customer.count())

## 3. Feature Engineering

In [ ]:
sales = (sales_clean
    .withColumn('Revenue', F.col('Quantity') * F.col('UnitPrice'))
    .withColumn('Date', F.to_date('InvoiceDate'))
    .withColumn('Year', F.year('InvoiceDate'))
    .withColumn('Month', F.month('InvoiceDate'))
    .withColumn('YearMonth', F.date_format('InvoiceDate', 'yyyy-MM'))
    .withColumn('DayOfWeek', F.date_format('InvoiceDate', 'E'))
    .withColumn('Hour', F.hour('InvoiceDate'))
)

sales.select('InvoiceNo','InvoiceDate','Quantity','UnitPrice','Revenue','YearMonth').show(10)

## 4. KPI tổng quan

In [ ]:
kpi = sales.agg(
    F.sum('Revenue').alias('Revenue'),
    F.countDistinct('InvoiceNo').alias('Orders'),
    F.sum('Quantity').alias('UnitsSold'),
    F.countDistinct('CustomerID').alias('Customers')
).first()

revenue = float(kpi['Revenue'])
orders = int(kpi['Orders'])
units = int(kpi['UnitsSold'])
customers = int(kpi['Customers'])

print(f'Revenue: £{revenue:,.2f}')
print(f'Orders: {orders:,}')
print(f'Units sold: {units:,}')
print(f'Customers: {customers:,}')
print(f'AOV: £{revenue/orders:,.2f}')
print(f'Revenue/customer: £{revenue/customers:,.2f}')
print(f'Items/order: {units/orders:,.2f}')

### Yêu cầu
Viết 3–5 câu nhận xét về quy mô hoạt động dựa trên KPI trên.

## 5. Phân tích doanh thu theo thời gian

In [ ]:
monthly = (sales.groupBy('YearMonth')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.sum('Quantity').alias('Units')
    )
    .withColumn('AOV', F.col('Revenue')/F.col('Orders'))
    .orderBy('YearMonth'))
monthly.show(30, truncate=False)

In [ ]:
import matplotlib.pyplot as plt
monthly_pd = monthly.toPandas()
plt.figure(figsize=(12,4))
plt.plot(monthly_pd['YearMonth'], monthly_pd['Revenue'], marker='o')
plt.xticks(rotation=90)
plt.title('Monthly Revenue')
plt.xlabel('Month'); plt.ylabel('Revenue (£)')
plt.tight_layout(); plt.show()

### Bài tập 3
- Tìm 10 ngày doanh thu cao nhất.
- Phân tích doanh thu theo thứ trong tuần.
- Phân tích theo giờ.
- Nhận xét mùa vụ/cao điểm.

In [ ]:
# TODO: doanh thu theo ngày / day-of-week / hour

## 6. Phân tích sản phẩm

In [ ]:
product_sales = (sales.groupBy('StockCode','Description')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.sum('Quantity').alias('Units'),
        F.countDistinct('InvoiceNo').alias('Orders')
    ))
product_sales.orderBy(F.desc('Revenue')).show(20, truncate=False)

### Bài tập 4
1. Top 20 theo Revenue.
2. Top 20 theo Units.
3. Top 20 theo Orders.
4. Tính tỷ trọng doanh thu Top 10/20/100.
5. Sản phẩm nào có Revenue cao nhưng Units không cao? Vì sao?

## 7. Phân tích thị trường theo quốc gia

In [ ]:
country_sales = (sales.groupBy('Country')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.countDistinct('CustomerID').alias('Customers')
    )
    .withColumn('AOV', F.col('Revenue')/F.col('Orders'))
    .orderBy(F.desc('Revenue')))
country_sales.show(20, truncate=False)

### Bài tập 5
- Top 10 quốc gia ngoài UK theo Revenue.
- So sánh AOV giữa các quốc gia có ≥ 50 orders.
- Thị trường quốc tế nào có tiềm năng nhất theo dữ liệu hiện có?

## 8. Phân tích khách hàng

In [ ]:
cust = (sales_customer
    .withColumn('Revenue', F.col('Quantity')*F.col('UnitPrice'))
    .groupBy('CustomerID')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.sum('Quantity').alias('Units')
    )
    .withColumn('AOV', F.col('Revenue')/F.col('Orders')))
cust.orderBy(F.desc('Revenue')).show(20)

### Bài tập 6. Concentration
Tính tỷ trọng Revenue do Top 1%, 5%, 10% khách hàng tạo ra. Nhận xét mức độ tập trung.

In [ ]:
# TODO: dùng Window để tính cumulative revenue share theo khách hàng

## 9. Phân tích RFM

In [ ]:
max_date = sales_customer.agg(F.max('InvoiceDate')).first()[0]
reference_date = max_date

rfm = (sales_customer
    .withColumn('Revenue', F.col('Quantity')*F.col('UnitPrice'))
    .groupBy('CustomerID')
    .agg(
        F.datediff(F.lit(reference_date), F.max('InvoiceDate')).alias('Recency'),
        F.countDistinct('InvoiceNo').alias('Frequency'),
        F.sum('Revenue').alias('Monetary')
    ))
rfm.show(10)

In [ ]:
# Chấm điểm 1–5 bằng ntile
w_r = Window.orderBy(F.col('Recency').desc())   # recency thấp sẽ nhận score cao
w_f = Window.orderBy(F.col('Frequency'))
w_m = Window.orderBy(F.col('Monetary'))

rfm_scored = (rfm
    .withColumn('R', F.ntile(5).over(w_r))
    .withColumn('F', F.ntile(5).over(w_f))
    .withColumn('M', F.ntile(5).over(w_m))
    .withColumn('RFMScore', F.concat_ws('', 'R','F','M')))
rfm_scored.show(10)

### Bài tập 7. Segmentation
Tạo cột `Segment` theo logic gợi ý:
- Champions: R≥4, F≥4, M≥4
- Loyal: F≥4 và R≥3
- Potential Loyalist: R≥4 và F=2–3
- At Risk: R≤2 và F≥3
- Lost/Other: còn lại

Sau đó tính số khách hàng, doanh thu và Monetary trung bình theo segment.

In [ ]:
rfm_segments = rfm_scored.withColumn(
    'Segment',
    F.when((F.col('R')>=4)&(F.col('F')>=4)&(F.col('M')>=4), 'Champions')
     .when((F.col('F')>=4)&(F.col('R')>=3), 'Loyal')
     .when((F.col('R')>=4)&(F.col('F').between(2,3)), 'Potential Loyalist')
     .when((F.col('R')<=2)&(F.col('F')>=3), 'At Risk')
     .otherwise('Lost/Other')
)

rfm_segments.groupBy('Segment').agg(
    F.count('*').alias('Customers'),
    F.sum('Monetary').alias('Revenue'),
    F.avg('Monetary').alias('AvgMonetary'),
    F.avg('Frequency').alias('AvgFrequency'),
    F.avg('Recency').alias('AvgRecency')
).orderBy(F.desc('Revenue')).show(truncate=False)

## 10. Spark SQL

In [ ]:
sales.createOrReplaceTempView('sales')

In [ ]:
spark.sql('''
SELECT YearMonth,
       ROUND(SUM(Revenue), 2) AS Revenue,
       COUNT(DISTINCT InvoiceNo) AS Orders
FROM sales
GROUP BY YearMonth
ORDER BY YearMonth
''').show(30)

### Bài tập 8. Viết thêm 5 truy vấn SQL
1. Top 10 sản phẩm theo Revenue.
2. Top 10 khách hàng theo Revenue.
3. Top 10 quốc gia ngoài UK.
4. Tháng có AOV cao nhất.
5. Sản phẩm xuất hiện trong nhiều hóa đơn nhất.

In [ ]:
# TODO: Spark SQL queries

## 11. Pareto 80/20

In [ ]:
total_product_revenue = product_sales.agg(F.sum('Revenue')).first()[0]
w = Window.orderBy(F.desc('Revenue')).rowsBetween(Window.unboundedPreceding, Window.currentRow)
pareto_products = (product_sales
    .orderBy(F.desc('Revenue'))
    .withColumn('CumRevenue', F.sum('Revenue').over(w))
    .withColumn('CumShare', F.col('CumRevenue') / F.lit(total_product_revenue)))
pareto_products.show(20, truncate=False)

### Câu hỏi
Bao nhiêu % sản phẩm tạo ra 80% doanh thu? Có thực sự là quy luật 80/20 không? Lặp lại phân tích cho khách hàng.

## 12. Executive Summary: Sinh viên phải hoàn thành
Viết ngắn gọn: 
1. **5 insights quan trọng nhất**.
2. **3 hành động kinh doanh** dựa trên dữ liệu.
3. **2 hạn chế của dataset/phân tích**.

> Mỗi insight phải có số liệu hoặc biểu đồ hỗ trợ, tránh nhận xét chung chung.

## 13. Câu hỏi thảo luận
1. Vì sao không nên `toPandas()` toàn bộ dataset?
2. Vì sao phải giữ riêng thông tin giao dịch hủy thay vì xóa ngay từ đầu?
3. RFM có hạn chế gì?
4. Nếu dữ liệu tăng lên 100 triệu dòng, đâu là lợi thế của Spark?
5. Nếu có thêm cost/profit, bài phân tích nên thay đổi thế nào?

# Kết thúc